# StayNest - Session 7 Assignment (Delta Lake & Lakehouse)
Work through the 8 tasks in order. Read the Assignment Questions PDF for the full
detail and acceptance criteria. Fill each `# TODO` cell, run it, and keep the output
visible. Runs on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `bookings_updates.csv` to a Volume, set `BASE`,
`CATALOG`, `SCHEMA`, and run this cell. Expect 12000 / 200 / 200.

In [0]:
BASE    = "/Volumes/workspace/default/staynest2"
CATALOG = "workspace"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

bookings: 12000, hotels: 200, updates: 200


## Task 1 - Read the plan and force a broadcast join
Join bookings to hotels and call `.explain()` to see the plan. Then force a
broadcast join with `broadcast(hotels_df)` and `.explain()` again. In a comment,
say which join each plan used and why broadcast avoids a shuffle.
(Tip: hotels also has a `city` column, so `hotels_df.drop("city")` before joining.)

In [0]:
from pyspark.sql.functions import broadcast

# Remove the duplicate city column from the hotel side.
hotels_for_join = hotels_df.drop("city")

# ---------------------------------------------------------
# 1. Let Catalyst choose the join strategy
# ---------------------------------------------------------
normal_join_df = bookings_df.join(
    hotels_for_join,
    on="hotel_id",
    how="inner"
)

print("===== PLAN CHOSEN BY CATALYST =====")
normal_join_df.explain(mode="formatted")


# ---------------------------------------------------------
# 2. Explicitly force hotels_df to be broadcast
# ---------------------------------------------------------
broadcast_join_df = bookings_df.join(
    broadcast(hotels_for_join),
    on="hotel_id",
    how="inner"
)

print("===== FORCED BROADCAST JOIN PLAN =====")
broadcast_join_df.explain(mode="formatted")


# Explanation:
# The first plan uses the join strategy Catalyst considers most appropriate.
# The forced plan uses BroadcastHashJoin because the small hotels table is
# copied to every executor, allowing each bookings partition to join locally
# without shuffling the large bookings table across the cluster.


===== PLAN CHOSEN BY CATALYST =====
== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonProject (11)
         +- PhotonBroadcastHashJoin Inner (10)
            :- PhotonFilter (3)
            :  +- PhotonRowToColumnar (2)
            :     +- Scan csv  (1)
            +- PhotonShuffleExchangeSource (9)
               +- PhotonShuffleMapStage (8)
                  +- PhotonShuffleExchangeSink (7)
                     +- PhotonFilter (6)
                        +- PhotonRowToColumnar (5)
                           +- Scan csv  (4)


(1) Scan csv 
Output [8]: [booking_id#11220, customer_id#11221, hotel_id#11222, booking_date#11223, city#11224, nights#11225, amount#11226, status#11227]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/staynest2/bookings.csv]
PushedFilters: [IsNotNull(hotel_id)]
ReadSchema: struct<booking_id:int,customer_id:int,hotel_id:int,booking_date:date,city:s

## Task 2 - Create a Delta table, then read its history
Write `bookings_df` as a managed Delta table with `saveAsTable`. Then create some
history: run an `UPDATE` (set pending to completed) and a `DELETE` (remove
cancelled). Show `DESCRIBE HISTORY` and point out the versioned commits.

In [0]:
# Fully qualified managed-table name
bookings_delta_table = FQN("bookings_delta")

# Start from a clean table so the history is easy to verify.
spark.sql(f"DROP TABLE IF EXISTS {bookings_delta_table}")

# ---------------------------------------------------------
# Version 0: WRITE
# Create a managed Delta table from bookings_df.
# ---------------------------------------------------------
(bookings_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bookings_delta_table))

print("Count after WRITE:", spark.table(bookings_delta_table).count())


# ---------------------------------------------------------
# Version 1: UPDATE
# Change every pending booking to completed.
# ---------------------------------------------------------
spark.sql(f"""
    UPDATE {bookings_delta_table}
    SET status = 'completed'
    WHERE status = 'pending'
""")

print("Remaining pending bookings:")
spark.sql(f"""
    SELECT COUNT(*) AS pending_count
    FROM {bookings_delta_table}
    WHERE status = 'pending'
""").show()


# ---------------------------------------------------------
# Version 2: DELETE
# Remove all cancelled bookings.
# ---------------------------------------------------------
spark.sql(f"""
    DELETE FROM {bookings_delta_table}
    WHERE status = 'cancelled'
""")

print("Count after DELETE:", spark.table(bookings_delta_table).count())


# ---------------------------------------------------------
# Display the Delta transaction history.
# ---------------------------------------------------------
history_df = spark.sql(f"""
    DESCRIBE HISTORY {bookings_delta_table}
""")

display(
    history_df
        .select(
            "version",
            "timestamp",
            "operation",
            "operationParameters",
            "operationMetrics"
        )
        .orderBy("version")
)


Count after WRITE: 12000
Remaining pending bookings:
+-------------+
|pending_count|
+-------------+
|            0|
+-------------+

Count after DELETE: 10563


version,timestamp,operation,operationParameters,operationMetrics
0,2026-07-20T23:39:46.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)","Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 12000, numOutputBytes -> 113163)"
1,2026-07-20T23:39:56.000Z,UPDATE,"Map(predicate -> [""(status#11672 = pending)""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 6637, numDeletionVectorsUpdated -> 0, scanTimeMs -> 3652, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 2936)"
2,2026-07-20T23:40:00.000Z,DELETE,"Map(predicate -> [""(status#12190 = cancelled)""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1990, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 1228, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 761)"


## Task 3 - Time travel and RESTORE
Read the table as it was at **version 0** (before your UPDATE and DELETE) and show
its count. Then `RESTORE` the table to version 0 and confirm the count is back.
Show that RESTORE appears as a new commit in the history.

In [0]:
# Fully qualified table name created in Task 2
bookings_delta_table = FQN("bookings_delta")

# ---------------------------------------------------------
# 1. Count the current table after UPDATE and DELETE
# ---------------------------------------------------------
current_count_before_restore = spark.table(
    bookings_delta_table
).count()

print(
    "Current count after UPDATE and DELETE:",
    current_count_before_restore
)


# ---------------------------------------------------------
# 2. Time travel: read the table exactly as it was at version 0
# ---------------------------------------------------------
version_0_df = (
    spark.read
        .format("delta")
        .option("versionAsOf", 0)
        .table(bookings_delta_table)
)

version_0_count = version_0_df.count()

print("Version 0 count:", version_0_count)
print(
    "Version 0 differs from current table:",
    version_0_count != current_count_before_restore
)


# ---------------------------------------------------------
# 3. Restore the current table to version 0
# RESTORE creates a new commit; it does not delete the history.
# ---------------------------------------------------------
restore_result = spark.sql(f"""
    RESTORE TABLE {bookings_delta_table}
    TO VERSION AS OF 0
""")

display(restore_result)


# ---------------------------------------------------------
# 4. Confirm that the restored table matches version 0
# ---------------------------------------------------------
count_after_restore = spark.table(
    bookings_delta_table
).count()

print("Count after RESTORE:", count_after_restore)
print(
    "Restored count matches version 0:",
    count_after_restore == version_0_count
)


# Optional validation checks
assert version_0_count != current_count_before_restore
assert count_after_restore == version_0_count


# ---------------------------------------------------------
# 5. Show that RESTORE was recorded as a new Delta commit
# ---------------------------------------------------------
history_df = spark.sql(f"""
    DESCRIBE HISTORY {bookings_delta_table}
""")

display(
    history_df
        .select(
            "version",
            "timestamp",
            "operation",
            "operationParameters",
            "operationMetrics"
        )
        .orderBy("version")
)


Current count after UPDATE and DELETE: 10563
Version 0 count: 12000
Version 0 differs from current table: True


table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
113163,1,1,1,101467,113163


Count after RESTORE: 12000
Restored count matches version 0: True


version,timestamp,operation,operationParameters,operationMetrics
0,2026-07-20T23:39:46.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)","Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 12000, numOutputBytes -> 113163)"
1,2026-07-20T23:39:56.000Z,UPDATE,"Map(predicate -> [""(status#11672 = pending)""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 6637, numDeletionVectorsUpdated -> 0, scanTimeMs -> 3652, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 2936)"
2,2026-07-20T23:40:00.000Z,DELETE,"Map(predicate -> [""(status#12190 = cancelled)""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1990, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 1228, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 761)"
3,2026-07-20T23:40:05.000Z,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)","Map(numRemovedFiles -> 2, numRemovedBytes -> 125764, p25FileSize -> 101467, numDeletionVectorsRemoved -> 1, minFileSize -> 101467, numAddedFiles -> 1, maxFileSize -> 101467, p75FileSize -> 101467, p50FileSize -> 101467, numAddedBytes -> 101467)"
4,2026-07-20T23:40:20.000Z,RESTORE,"Map(version -> 0, timestamp -> null)","Map(numRestoredFiles -> 1, removedFilesSize -> 101467, numRemovedFiles -> 1, restoredFilesSize -> 113163, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 113163)"


## Task 4 - OPTIMIZE and ZORDER
Run `OPTIMIZE` on your Delta table to compact files. Then run
`OPTIMIZE ... ZORDER BY (city)`. In a comment, say what OPTIMIZE does and why
`city` is a good ZORDER column but `status` would not be.

In [0]:
# Fully qualified Delta table created in Task 2
bookings_delta_table = FQN("bookings_delta")

# ---------------------------------------------------------
# 1. Compact small files into fewer, larger files
# ---------------------------------------------------------
optimize_result = spark.sql(f"""
    OPTIMIZE {bookings_delta_table}
""")

display(optimize_result)


# ---------------------------------------------------------
# 2. Compact the files and reorganize rows by city
# ---------------------------------------------------------
zorder_result = spark.sql(f"""
    OPTIMIZE {bookings_delta_table}
    ZORDER BY (city)
""")

display(zorder_result)


# OPTIMIZE compacts many small Delta files into fewer, larger files,
# reducing file-opening overhead. city is suitable for ZORDER because
# queries commonly filter by city and it has more distinct values;
# status has low cardinality, so clustering by it provides little data skipping.


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1784590829623, 1784590831973, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null)"


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 113163), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1784590834050, 1784590837502, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null)"


## Task 5 - Bronze: land the raw data
Write the raw bookings to a `bronze_bookings` Delta table, keeping every row and
adding an `ingested_at` timestamp column.

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_table = FQN("bronze_bookings")

# Start from a clean table so the cell can be rerun safely.
spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")

# Keep every raw booking row and add ingestion metadata.
bronze_bookings_df = (
    bookings_df
        .withColumn("ingested_at", current_timestamp())
)

# Write as a managed Delta table.
(
    bronze_bookings_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(bronze_table)
)

# Read the persisted table back for validation.
bronze_check_df = spark.table(bronze_table)

bronze_count = bronze_check_df.count()

print("Bronze table:", bronze_table)
print("Source row count:", bookings_df.count())
print("Bronze row count:", bronze_count)
print("Has ingested_at column:", "ingested_at" in bronze_check_df.columns)

bronze_check_df.printSchema()
display(bronze_check_df.limit(10))

# Acceptance checks
assert bronze_count == 12_000
assert bronze_count == bookings_df.count()
assert "ingested_at" in bronze_check_df.columns


Bronze table: workspace.default.bronze_bookings
Source row count: 12000
Bronze row count: 12000
Has ingested_at column: True
root
 |-- booking_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- hotel_id: integer (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- nights: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



booking_id,customer_id,hotel_id,booking_date,city,nights,amount,status,ingested_at
9000000,701600,3095,2025-11-27,Jaipur,4,6087.65,completed,2026-07-20T23:40:41.511Z
9000001,700065,3057,2025-11-06,Delhi,1,8211.19,cancelled,2026-07-20T23:40:41.511Z
9000002,701392,3187,2025-08-21,Jaipur,2,7176.52,cancelled,2026-07-20T23:40:41.511Z
9000003,700867,3112,2025-03-22,Bengaluru,5,7880.62,completed,2026-07-20T23:40:41.511Z
9000004,701521,3043,2025-04-19,Mumbai,5,21021.51,pending,2026-07-20T23:40:41.511Z
9000005,701998,3018,2025-10-14,Mumbai,2,18124.49,cancelled,2026-07-20T23:40:41.511Z
9000006,701336,3012,2025-11-25,Delhi,7,70999.15,completed,2026-07-20T23:40:41.511Z
9000007,700868,3127,2025-04-10,Mumbai,2,15693.64,completed,2026-07-20T23:40:41.511Z
9000008,700687,3045,2025-01-15,Mumbai,1,1492.62,completed,2026-07-20T23:40:41.511Z
9000009,700022,3040,2025-06-08,Goa,5,6694.39,completed,2026-07-20T23:40:41.511Z


## Task 6 - Silver: clean and conform
Build `silver_bookings` from bronze: keep only completed bookings and join the
hotel dimension to add `category`, `star_rating`, and the hotel name. Drop the
duplicate `city` from the hotel side so the join has a single `city`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

bronze_table = FQN("bronze_bookings")
silver_table = FQN("silver_bookings")

# Read the persisted Bronze table.
bronze_df = spark.table(bronze_table)

# Exclude hotels.city so Silver contains only one city column.
hotels_for_silver = hotels_df.select(
    "hotel_id",
    "hotel_name",
    "category",
    "star_rating"
)

# Keep completed bookings, enrich them, and cast amount to DecimalType.
silver_bookings_df = (
    bronze_df
        .filter(F.col("status") == "completed")
        .join(
            hotels_for_silver,
            on="hotel_id",
            how="inner"
        )
        .withColumn(
            "amount",
            F.col("amount").cast("decimal(18,2)")
        )
)

print("Schema before writing:")
silver_bookings_df.select("amount").printSchema()

# Replace the table data and its stored schema.
(
    silver_bookings_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
)

# Do not use REFRESH TABLE on serverless.
# Create a new DataFrame after the write.
silver_check_df = spark.table(silver_table)

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------
silver_count = silver_check_df.count()

completed_bronze_count = (
    bronze_df
        .filter(F.col("status") == "completed")
        .count()
)

print("Silver table:", silver_table)
print("Completed bookings in Bronze:", completed_bronze_count)
print("Rows in Silver:", silver_count)
print("Number of city columns:", silver_check_df.columns.count("city"))

silver_check_df.selectExpr(
    "typeof(amount) AS amount_type"
).show()

silver_check_df.printSchema()

display(
    silver_check_df.select(
        "booking_id",
        "hotel_id",
        "hotel_name",
        "city",
        "category",
        "star_rating",
        "amount",
        "status",
        "ingested_at"
    ).limit(10)
)

# Acceptance checks
assert silver_count == completed_bronze_count
assert silver_check_df.columns.count("city") == 1
assert silver_check_df.filter(F.col("status") != "completed").count() == 0

for required_column in ["hotel_name", "category", "star_rating"]:
    assert required_column in silver_check_df.columns

assert isinstance(
    silver_check_df.schema["amount"].dataType,
    DecimalType
), "Silver amount is still not DecimalType"


Schema before writing:
root
 |-- amount: decimal(18,2) (nullable = true)

Silver table: workspace.default.silver_bookings
Completed bookings in Bronze: 9660
Rows in Silver: 9660
Number of city columns: 1
+-------------+
|  amount_type|
+-------------+
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
+-------------+
only showing top 20 rows
root
 |-- hotel_id: integer (nullable = true)
 |-- booking_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- nights: integer (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- hot

booking_id,hotel_id,hotel_name,city,category,star_rating,amount,status,ingested_at
9000000,3095,Orchid Suites,Jaipur,Budget,3.8,6087.65,completed,2026-07-20T23:40:41.511Z
9000003,3112,Grand Stay,Bengaluru,Budget,3.6,7880.62,completed,2026-07-20T23:40:41.511Z
9000006,3012,Orchid Residency,Delhi,Luxury,4.1,70999.15,completed,2026-07-20T23:40:41.511Z
9000007,3127,Summit Stay,Mumbai,Luxury,4.5,15693.64,completed,2026-07-20T23:40:41.511Z
9000008,3045,Sunset Inn,Mumbai,Budget,3.0,1492.62,completed,2026-07-20T23:40:41.511Z
9000009,3040,Amber Grand,Goa,Budget,3.7,6694.39,completed,2026-07-20T23:40:41.511Z
9000010,3054,Riverside Resort,Goa,Premium,4.9,30786.36,completed,2026-07-20T23:40:41.511Z
9000011,3168,Coral Retreat,Bengaluru,Budget,4.3,1303.83,completed,2026-07-20T23:40:41.511Z
9000012,3195,Summit Stay,Munnar,Premium,4.6,15542.04,completed,2026-07-20T23:40:41.511Z
9000013,3071,Palm Resort,Jaipur,Luxury,2.7,60473.23,completed,2026-07-20T23:40:41.511Z


## Task 7 - Gold: business-ready aggregate
From silver, build a `gold_city_revenue` Delta table: bookings and total revenue
per city, ordered by revenue.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

silver_table = FQN("silver_bookings")
gold_table = FQN("gold_city_revenue")

# Refresh and read the corrected persisted Silver table.
silver_df = spark.table(silver_table)

# Confirm Silver is using DecimalType before aggregation.
silver_df.selectExpr(
    "typeof(amount) AS amount_type"
).show()

assert isinstance(
    silver_df.schema["amount"].dataType,
    DecimalType
), "Run the corrected Silver cell first: amount is not DecimalType"


# ---------------------------------------------------------
# Build the Gold city-level aggregate
# ---------------------------------------------------------
gold_city_revenue_df = (
    silver_df
        .groupBy("city")
        .agg(
            F.count("*").alias("bookings"),
            F.sum("amount")
                .cast("decimal(28,2)")
                .alias("total_revenue")
        )
        .orderBy(F.col("total_revenue").desc())
)

print("Gold schema before writing:")
gold_city_revenue_df.printSchema()

# Overwrite both the existing data and the existing Delta schema.
(
    gold_city_revenue_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(gold_table)
)

# Refresh and re-read the persisted Gold table.
gold_check_df = spark.table(gold_table)

# ---------------------------------------------------------
# Validation and ordered output
# ---------------------------------------------------------
print("Gold table:", gold_table)
print("Number of cities:", gold_check_df.count())

gold_check_df.selectExpr(
    "typeof(total_revenue) AS revenue_type"
).show()

gold_check_df.printSchema()

display(
    gold_check_df.orderBy(
        F.col("total_revenue").desc()
    )
)

# Acceptance checks
assert "city" in gold_check_df.columns
assert "bookings" in gold_check_df.columns
assert "total_revenue" in gold_check_df.columns

assert isinstance(
    gold_check_df.schema["total_revenue"].dataType,
    DecimalType
), "Gold total_revenue is still not DecimalType"

assert (
    gold_check_df
        .filter(
            F.col("bookings").isNull()
            | F.col("total_revenue").isNull()
        )
        .count()
    == 0
)


# ---------------------------------------------------------
# Reconcile Silver and Gold totals
# ---------------------------------------------------------
silver_totals = (
    silver_df
        .agg(
            F.count("*").alias("bookings"),
            F.sum("amount")
                .cast("decimal(38,2)")
                .alias("revenue")
        )
        .first()
)

gold_totals = (
    gold_check_df
        .agg(
            F.sum("bookings").alias("bookings"),
            F.sum("total_revenue")
                .cast("decimal(38,2)")
                .alias("revenue")
        )
        .first()
)

silver_booking_count = silver_totals["bookings"]
gold_booking_count = gold_totals["bookings"]

silver_revenue = silver_totals["revenue"]
gold_revenue = gold_totals["revenue"]

print("Silver booking count:", silver_booking_count)
print("Gold booking count:", gold_booking_count)
print("Silver total revenue:", silver_revenue)
print("Gold total revenue:", gold_revenue)
print("Revenue difference:", silver_revenue - gold_revenue)

assert silver_booking_count == gold_booking_count
assert silver_revenue == gold_revenue

+-------------+
|  amount_type|
+-------------+
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
|decimal(18,2)|
+-------------+
only showing top 20 rows
Gold schema before writing:
root
 |-- city: string (nullable = true)
 |-- bookings: long (nullable = false)
 |-- total_revenue: decimal(28,2) (nullable = true)

Gold table: workspace.default.gold_city_revenue
Number of cities: 10
+-------------+
| revenue_type|
+-------------+
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
|decimal(28,2)|
+-------------+

root
 |-- city: string (nullable = true)
 |-- bookings: long (nullable = true)
 |-- total_revenue: decimal(28,2) (nullable = true)



city,bookings,total_revenue
Goa,2546,44596701.79
Mumbai,1715,36241221.12
Delhi,1174,26314281.54
Jaipur,979,24436853.13
Bengaluru,1318,22670136.97
Udaipur,691,12094427.42
Rishikesh,407,8606121.58
Manali,480,6235480.68
Munnar,244,3979216.11
Anantapur,106,2257080.65


Silver booking count: 9660
Gold booking count: 9660
Silver total revenue: 187431520.99
Gold total revenue: 187431520.99
Revenue difference: 0.00


In [0]:
print("Silver total revenue:", silver_revenue)
print("Gold total revenue:", gold_revenue)
print("Difference:", silver_revenue - gold_revenue)

Silver total revenue: 187431520.99
Gold total revenue: 187431520.99
Difference: 0.00


## Task 8 - Incremental load with MERGE
You have today's batch in `updates_df` (150 changed bookings + 50 new ones).
`MERGE` it into your Delta table: update matched booking_ids, insert new ones, in
one command. Report the row count before and after (it should grow by the 50 new).

In [0]:
bookings_delta_table = FQN("bookings_delta")

# Register today's batch as a temporary SQL view.
updates_df.createOrReplaceTempView("bookings_updates_source")

# Count the target before the incremental load.
before_count = spark.table(bookings_delta_table).count()

print("Rows before MERGE:", before_count)


# ---------------------------------------------------------
# One atomic MERGE:
# - update rows whose booking_id already exists
# - insert rows whose booking_id is new
# ---------------------------------------------------------
spark.sql(f"""
    MERGE INTO {bookings_delta_table} AS target
    USING bookings_updates_source AS source
        ON target.booking_id = source.booking_id

    WHEN MATCHED THEN
        UPDATE SET
            target.customer_id = source.customer_id,
            target.hotel_id = source.hotel_id,
            target.booking_date = source.booking_date,
            target.city = source.city,
            target.nights = source.nights,
            target.amount = source.amount,
            target.status = source.status

    WHEN NOT MATCHED THEN
        INSERT (
            booking_id,
            customer_id,
            hotel_id,
            booking_date,
            city,
            nights,
            amount,
            status
        )
        VALUES (
            source.booking_id,
            source.customer_id,
            source.hotel_id,
            source.booking_date,
            source.city,
            source.nights,
            source.amount,
            source.status
        )
""")


# Count the target after the incremental load.
after_count = spark.table(bookings_delta_table).count()

print("Rows after MERGE:", after_count)
print("Rows added:", after_count - before_count)

# Acceptance checks
assert after_count == before_count + 50
assert after_count - before_count == 50


Rows before MERGE: 12000
Rows after MERGE: 12050
Rows added: 50
